In [5]:
!pip install transformers torch

In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load pretrained conversational model with a larger Flan-T5 model
model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print(f"Chatbot initialized with {model_name}. Type 'exit' to end the conversation.\n")

conversation_history = []

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Chatbot: Goodbye! Have a nice day.")
        break

    # Add user input to conversation history (for conversational context)
    conversation_history.append(f"User: {user_input}")

    # Construct the prompt for the T5 model with an explicit instruction for the latest input
    # Take all previous turns as context
    context_turns = conversation_history[:-1] # All turns except the very last user input
    latest_user_question = conversation_history[-1].replace("User: ", "") # Extract the actual question

    # Combine context with the new instructed question
    prompt_for_model = "\n".join(context_turns)
    if prompt_for_model: # If there's previous conversation, add a newline
        prompt_for_model += "\n"
    prompt_for_model += f"Answer the following question: {latest_user_question}\nChatbot:"

    # Encode the prompt
    input_ids = tokenizer.encode(prompt_for_model, return_tensors="pt")

    # Generate response
    output_ids = model.generate(
        input_ids,
        max_length=512, # Increased max_length for longer responses
        num_beams=5, # Use beam search for better quality
        early_stopping=True
    )

    # Decode response
    response = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print("Chatbot:", response)

    # Add bot response to conversation history
    conversation_history.append(f"Chatbot: {response}")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Chatbot initialized with google/flan-t5-large. Type 'exit' to end the conversation.

You: What is Artificial Intelligence?
Chatbot: Artificial Intelligence (AI) is the use of artificial intelligence (AI) to improve the performance of human beings.
You: exit
Chatbot: Goodbye! Have a nice day.
